# PySpark Interview Self-Test #1

**Calibration:** Easy → Medium. Targets PySpark DataFrame API patterns frequently asked in service-company interviews (TCS / Infosys / Wipro / Capgemini / Accenture / Celebal tier) and client interviews on Databricks/EMR projects.

**Coverage:**
1. DataFrame basics — `select`, `filter`, `withColumn`, `orderBy`
2. GroupBy + multiple aggregations
3. Conditional column with `when` / `otherwise` (the CASE WHEN of PySpark)
4. Join with `broadcast` hint
5. Window function — `dense_rank` top-N per group
6. Window function — `lag` for month-over-month
7. Collections — `collect_set` / conditional aggregation in one `groupBy`
8. Handling nulls — self-join + `coalesce`
9. Pivot — `groupBy().pivot().agg()`
10. Repartition vs coalesce + partitioned write *(knowledge + code question)*

**Format:** markdown question → empty code cell. Use `.show()` to display results (the canonical interview style).

**Prereqs:** `pip install pyspark` plus Java 11+ in PATH (WSL: `sudo apt install openjdk-17-jdk-headless`). Same 3 tables as the SQL test (departments, employees, orders) — customers isn't needed.

When done, paste your 10 answers in chat for grading (chat-only — SQL/PySpark drills aren't archived per your study-material rule).

## Setup

Run once. Creates a local SparkSession and three DataFrames: `departments`, `employees`, `orders`.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import date

spark = (
    SparkSession.builder
    .appName('PySpark-Interview-Test-01')
    .master('local[*]')
    .config('spark.sql.shuffle.partitions', '4')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('ERROR')

departments = spark.createDataFrame(
    [
        (1, 'Engineering'),
        (2, 'Sales'),
        (3, 'Marketing'),
        (4, 'HR'),
        (5, 'Finance'),
    ],
    ['dept_id', 'dept_name'],
)

employees = spark.createDataFrame(
    [
        (1,  'Anjali',  120000, 1, None, date(2022, 1, 15)),
        (2,  'Rohit',    95000, 1, 1,    date(2022, 3, 20)),
        (3,  'Priya',   105000, 1, 1,    date(2022, 6, 10)),
        (4,  'Vikram',  130000, 1, 1,    date(2023, 1, 5)),
        (5,  'Suresh',   85000, 2, 6,    date(2022, 2, 15)),
        (6,  'Meera',    90000, 2, None, date(2021, 11, 1)),
        (7,  'Karan',    95000, 2, 6,    date(2023, 3, 15)),
        (8,  'Neha',     70000, 3, None, date(2022, 8, 20)),
        (9,  'Arjun',    65000, 3, 8,    date(2023, 5, 10)),
        (10, 'Kavita',   75000, 5, None, date(2023, 9, 1)),
    ],
    ['emp_id', 'name', 'salary', 'dept_id', 'manager_id', 'hire_date'],
)

orders = spark.createDataFrame(
    [
        (1,  1, date(2025, 1, 15),  5000.00, 'completed'),
        (2,  1, date(2025, 1, 16),  3000.00, 'completed'),
        (3,  1, date(2025, 1, 17),  2000.00, 'cancelled'),
        (4,  2, date(2025, 1, 20),  4500.00, 'completed'),
        (5,  3, date(2025, 2, 1),   6000.00, 'completed'),
        (6,  3, date(2025, 2, 2),   1500.00, 'pending'),
        (7,  4, date(2025, 2, 10),  8000.00, 'completed'),
        (8,  4, date(2025, 2, 11),  2500.00, 'completed'),
        (9,  1, date(2025, 2, 15),  3500.00, 'completed'),
        (10, 5, date(2025, 3, 5),  10000.00, 'completed'),
        (11, 5, date(2025, 3, 6),   4000.00, 'cancelled'),
        (12, 5, date(2025, 3, 7),   2500.00, 'completed'),
        (13, 5, date(2025, 3, 8),   1500.00, 'completed'),
        (14, 6, date(2025, 3, 10),  7000.00, 'pending'),
        (15, 2, date(2025, 3, 15),  5500.00, 'completed'),
        (16, 7, date(2025, 4, 1),   9000.00, 'completed'),
        (17, 7, date(2025, 4, 5),   3000.00, 'cancelled'),
        (18, 8, date(2025, 4, 10), 12000.00, 'completed'),
        (19, 3, date(2025, 4, 15),  4000.00, 'completed'),
        (20, 1, date(2025, 4, 20),  2000.00, 'completed'),
        (21, 4, date(2025, 5, 1),   6500.00, 'completed'),
        (22, 6, date(2025, 5, 3),   8500.00, 'completed'),
        (23, 6, date(2025, 5, 4),   1000.00, 'pending'),
        (24, 2, date(2025, 5, 10),  5000.00, 'completed'),
        (25, 8, date(2025, 5, 15),  7500.00, 'cancelled'),
    ],
    ['order_id', 'cust_id', 'order_date', 'amount', 'status'],
)

print('Setup complete. DataFrames: departments, employees, orders.')

ModuleNotFoundError: No module named 'pyspark'

## Sample data preview

In [ ]:
departments.show()

In [ ]:
employees.orderBy('dept_id', F.desc('salary')).show()

In [ ]:
orders.orderBy('order_date').show(30)

---
## NQ1 — DataFrame basics: select / filter / withColumn / orderBy

From `employees`, return:
- Employees in **Engineering (dept_id = 1)** only
- Columns: `name`, `salary`, `salary_in_lakhs` (a new column equal to `salary / 100000`)
- Sorted by `salary` descending

Use `.show()` to display.

In [ ]:
# NQ1



---
## NQ2 — GroupBy + multiple aggregations

For each `dept_id` in `employees`, return: `emp_count`, `total_salary`, `avg_salary` (rounded to 2 decimals), `max_salary`.

Return ONE row per department in a single `.groupBy().agg()` call. Order by `dept_id`.

In [ ]:
# NQ2



---
## NQ3 — Conditional column with `when` / `otherwise`

Add a column `salary_band` to `employees`:
- `'HIGH'` if salary >= 100000
- `'MID'` if salary between 80000 and 99999
- `'LOW'` otherwise

Return `name`, `salary`, `salary_band`. Order by `salary` descending.

*This is the PySpark equivalent of SQL `CASE WHEN`. Chain `when` calls and finish with `.otherwise(...)`.*

In [ ]:
# NQ3



---
## NQ4 — Join with `broadcast` hint

Join `employees` with `departments` to produce a DataFrame with columns: `name`, `salary`, `dept_name`. Use a **broadcast hint** since `departments` is small (~5 rows) — this avoids a shuffle.

Order by `dept_name`, then `salary` descending.

*Recall the rule from your DE_Interview_Prep.md: BroadcastHashJoin works when one side fits in driver+executor memory. 5-row dim qualifies.*

In [ ]:
# NQ4



---
## NQ5 — Window function: top 2 salary tiers per department (DENSE_RANK)

For each department, return employees in the **top 2 salary tiers** (ties share a tier, next distinct salary is the following tier).

Return: `dept_name`, `name`, `salary`, `salary_tier`. Order by `dept_name`, `salary_tier`, `name`.

*Use `Window.partitionBy(...).orderBy(...)` and `F.dense_rank()`. Filter by tier with `.filter(F.col('salary_tier') <= 2)` — this is a row-level filter, not a HAVING.*

In [ ]:
# NQ5



---
## NQ6 — Window function: month-over-month growth %

**From completed orders only**, compute each month's total revenue and the month-over-month % change. Round to 2 decimals.

Return: `month` (use `F.date_trunc('month', order_date)` or `F.trunc('month', order_date)`), `revenue`, `mom_growth_pct`. Order by `month`. First month's `mom_growth_pct` should be NULL.

*Two parts: (1) filter completed + aggregate per month, (2) `F.lag('revenue')` over `Window.orderBy('month')` to get previous month's revenue, then compute growth.*

In [ ]:
# NQ6



---
## NQ7 — Collections: distinct status list + conditional count, one groupBy

For each `cust_id` in `orders`, return in a **single `.groupBy('cust_id').agg(...)`** call:
- `distinct_statuses` — an array of that customer's distinct order statuses, sorted alphabetically
- `completed_count` — count of their completed orders

Order by `cust_id`.

*Use `F.collect_set('status')` for the distinct-array, then `F.array_sort(...)` to sort it. For the conditional count, use `F.sum(F.when(F.col('status') == 'completed', 1).otherwise(0))` — same `SUM(CASE WHEN ...)` pattern as SQL, just spelled in PySpark.*

In [ ]:
# NQ7



---
## NQ8 — Self-join + null handling: employee → manager name

From `employees`, return: `name`, `salary`, `manager_name` — where `manager_name` is the name of the employee's manager (look up `manager_id` → `emp_id`).

**For employees with no manager (manager_id is NULL), `manager_name` must show `'NO MANAGER'` — not `null`.**

Order by `name`.

*Use a LEFT JOIN of `employees` to itself (alias the two sides — `e` and `m`). Then `F.coalesce(m_name, F.lit('NO MANAGER'))`. Or use `.na.fill({'manager_name': 'NO MANAGER'})` after the join.*

In [ ]:
# NQ8



---
## NQ9 — Pivot: status counts per month

For each month in `orders`, return one row per month with columns: `completed`, `cancelled`, `pending` — each holding the count of that status in that month.

Return: `month`, `completed`, `cancelled`, `pending`. Order by `month`. Replace any null counts with 0.

*Use `groupBy('month').pivot('status', ['completed', 'cancelled', 'pending']).count()`. Explicitly listing the values in `pivot(...)` is the production-safe pattern — it avoids a hidden scan to discover values.*

In [ ]:
# NQ9



---
## NQ10 — Repartition vs coalesce + partitioned write *(knowledge + code)*

**Scenario:** You have a DataFrame `sales` with 200 input partitions, ~5 GB total. You want to write it as Parquet, partitioned on disk by `year_month`, with each output file ~128 MB.

**Part A — Code.** Write the PySpark code to do this. Assume `sales` already has a `year_month` column (string like `'2025-01'`).

**Part B — Write in a markdown comment inside the code cell:** *Why would you use `repartition` here rather than `coalesce`? When is `coalesce` the right call instead?*

*Hints (allowed because this is a knowledge question, not a drill):*
- `repartition(n, col)` triggers a shuffle and balances partitions evenly — expensive but correct for skew.
- `coalesce(n)` reduces partition count WITHOUT a shuffle by merging neighbors — cheap, but can produce skewed partitions.
- For partitioned writes, you typically want one writer task per output partition value to avoid the small-files problem AND skew.

In [ ]:
# NQ10

# Part A code:


# Part B answer (in this comment):
# Why repartition over coalesce here?
# 
# When would coalesce be the right call instead?
# 

---
## Self-verify checklist (PySpark gotchas)

- **NQ1:** Did you use `F.col('salary') / 100000` (not just `'salary' / 100000` — that's a Python string divide)?
- **NQ2:** All four aggregates in ONE `.agg(...)` call (not chained `.agg().agg()`)? `F.round(F.avg(...), 2)` applied correctly?
- **NQ3:** Chained `F.when(...).when(...).otherwise(...)` — NOT separate `.withColumn` calls per band? The `when` chain reads top-down, first match wins.
- **NQ4:** Did you import `from pyspark.sql.functions import broadcast` (or `F.broadcast`)? Inspect with `.explain()` — should see `BroadcastHashJoin`, not `SortMergeJoin`.
- **NQ5:** Filter is `.filter(F.col('salary_tier') <= 2)` — a row filter on the window-computed col. Engineering should return 2 rows (Vikram tier 1, Anjali tier 2 — Priya is tier 3 since no tie in this dataset).
- **NQ6:** Did you filter `status == 'completed'` BEFORE aggregating? First row's `mom_growth_pct` must be NULL (LAG returns null automatically).
- **NQ7:** ONE groupBy. The `collect_set` is unordered — wrap in `F.array_sort(...)` to make the test deterministic.
- **NQ8:** Did you alias both sides of the self-join? `employees.alias('e').join(employees.alias('m'), ...)`. Did `coalesce` (or `na.fill`) replace null with `'NO MANAGER'`?
- **NQ9:** Did you pass the explicit value list to `pivot('status', [...])`? Otherwise Spark does an extra scan to discover values — wasteful in real pipelines.
- **NQ10:** Did your repartition argument match the partition column used in `.write.partitionBy(...)`? Mismatched cols = small-files problem returns.

When done, paste your 10 answers in chat for grading.